# Módulos estándar · Expresiones regulares (`re`)

Buscar, extraer y limpiar texto con patrones.

## 1. Palabras más frecuentes

Obtener las palabras del párrafo con `re.findall` y listar cada una con su frecuencia, de más a menos frecuente.

In [1]:
import re
from collections import Counter

paragraph = ("I love teaching. If you do not love teaching what else can you love. I love Python if you do not "
             "love something which can give you all the capabilities to develop an application what else can you love.")
frecuencias = Counter(re.findall(r"\w+", paragraph))
print([(n, palabra) for palabra, n in frecuencias.most_common()])

[(6, 'love'), (5, 'you'), (3, 'can'), (2, 'I'), (2, 'teaching'), (2, 'do'), (2, 'not'), (2, 'what'), (2, 'else'), (1, 'If'), (1, 'Python'), (1, 'if'), (1, 'something'), (1, 'which'), (1, 'give'), (1, 'all'), (1, 'the'), (1, 'capabilities'), (1, 'to'), (1, 'develop'), (1, 'an'), (1, 'application')]


## 2. Partículas en el eje x

Extraer del texto las posiciones de las partículas y calcular la distancia entre las dos más alejadas.

In [2]:
texto = ("The position of some particles on the horizontal x-axis are -12, -4, -3 and -1 in the negative "
         "direction, 0 at origin, 4 and 8 in the positive direction.")
# (?<!\w) evita tomar el '-' de 'x-axis'
puntos = sorted(int(n) for n in re.findall(r"(?<!\w)-?\d+", texto))
print(puntos, puntos[-1] - puntos[0])

[-12, -4, -3, -1, 0, 4, 8] 20


## 3. Nombre de variable válido

Escribir con un patrón `is_valid_variable`, que diga si una cadena es un nombre de variable válido en Python.

In [3]:
def is_valid_variable(nombre):
    return re.fullmatch(r"[A-Za-z_]\w*", nombre) is not None

for nombre in ("first_name", "first-name", "1first_name", "firstname"):
    print(nombre, is_valid_variable(nombre))

first_name True
first-name False
1first_name False
firstname True


## 4. Limpiar texto

Eliminar los símbolos sobrantes del texto y obtener las tres palabras más frecuentes.

In [4]:
sentence = """%I $am@% a %tea@cher%, &and& I lo%#ve %tea@ching%;. There $is nothing; &as& mo@re rewarding as educa@ting &and& @emp%o@wering peo@ple. ;I found tea@ching m%o@re interesting tha@n any other %jo@bs. %Do@es thi%s mo@tivate yo@u to be a tea@cher!?"""

def clean_text(texto):
    return re.sub(r"[^\w\s]", "", texto)

def most_frequent_words(texto, n=3):
    return [(c, p) for p, c in Counter(texto.split()).most_common(n)]

limpio = clean_text(sentence)
print(limpio)
print(most_frequent_words(limpio))

I am a teacher and I love teaching There is nothing as more rewarding as educating and empowering people I found teaching more interesting than any other jobs Does this motivate you to be a teacher
[(3, 'I'), (2, 'a'), (2, 'teacher')]


## 5. Grupos con nombre

Analizar las líneas de un registro con grupos con nombre (`(?P<nombre>...)`), obtener un diccionario por línea con `groupdict` y descartar las líneas que no encajan.

In [5]:
registro = """2026-11-06 09:15:02 INFO [web] Petición atendida en 35 ms
2026-11-06 09:15:07 ERROR [db] Tiempo de espera agotado
línea corrupta
2026-11-06 09:16:40 WARNING [auth] Contraseña incorrecta para admin"""

LINEA = re.compile(r"(?P<fecha>\d{4}-\d{2}-\d{2}) (?P<hora>[\d:]{8}) (?P<nivel>[A-Z]+) \[(?P<modulo>\w+)\] (?P<mensaje>.*)")

entradas = [m.groupdict() for m in map(LINEA.fullmatch, registro.splitlines()) if m]
for entrada in entradas:
    print(entrada)
assert [e["nivel"] for e in entradas] == ["INFO", "ERROR", "WARNING"]
assert entradas[1]["modulo"] == "db"

{'fecha': '2026-11-06', 'hora': '09:15:02', 'nivel': 'INFO', 'modulo': 'web', 'mensaje': 'Petición atendida en 35 ms'}
{'fecha': '2026-11-06', 'hora': '09:15:07', 'nivel': 'ERROR', 'modulo': 'db', 'mensaje': 'Tiempo de espera agotado'}
{'fecha': '2026-11-06', 'hora': '09:16:40', 'nivel': 'WARNING', 'modulo': 'auth', 'mensaje': 'Contraseña incorrecta para admin'}


## 6. `findall`, `finditer` y `sub` con función

En una receta, listar las cantidades con `findall`, mostrar su posición con `finditer` y duplicarlas todas con `sub` pasándole una función.

In [6]:
receta = "Mezclar 250 g de harina, 2 huevos, 125 ml de leche y 1.5 cucharadas de azúcar."
CANTIDAD = re.compile(r"\d+(?:\.\d+)?")

print(CANTIDAD.findall(receta))
for m in CANTIDAD.finditer(receta):
    print(f"{m.group():>4} en {m.span()}")

def doble(m):
    return f"{float(m.group()) * 2:g}"

doble_receta = CANTIDAD.sub(doble, receta)
print(doble_receta)
assert doble_receta == "Mezclar 500 g de harina, 4 huevos, 250 ml de leche y 3 cucharadas de azúcar."

['250', '2', '125', '1.5']
 250 en (8, 11)
   2 en (25, 26)
 125 en (35, 38)
 1.5 en (53, 56)
Mezclar 500 g de harina, 4 huevos, 250 ml de leche y 3 cucharadas de azúcar.


## 7. DNI y NIE

Escribir `es_dni_valido`, que acepte un DNI (8 dígitos y letra) o un NIE (`X`, `Y` o `Z`, 7 dígitos y letra) y compruebe la letra de control: el resto del número entre 23 indexa la cadena `TRWAGMYFPDXBNJZSQVHLCKE`, y en el NIE la letra inicial vale 0, 1 o 2.

In [7]:
LETRAS = "TRWAGMYFPDXBNJZSQVHLCKE"
DOCUMENTO = re.compile(r"(?P<numero>\d{8}|[XYZ]\d{7})-?(?P<letra>[A-Z])")

def es_dni_valido(texto):
    m = DOCUMENTO.fullmatch(texto.strip().upper())
    if not m:
        return False
    numero = int(m["numero"].translate(str.maketrans("XYZ", "012")))
    return LETRAS[numero % 23] == m["letra"]

casos = {"12345678Z": True, "12345678-z": True, "12345678A": False, "X1234567L": True,
         "Y1234567X": True, "1234567Z": False, "W1234567L": False}
for documento, esperado in casos.items():
    print(f"{documento:>11}: {es_dni_valido(documento)}")
    assert es_dni_valido(documento) == esperado

  12345678Z: True
 12345678-z: True
  12345678A: False
  X1234567L: True
  Y1234567X: True
   1234567Z: False
  W1234567L: False


## 8. Emails

Validar direcciones de correo con un patrón sencillo (parte local, `@` y dominio con al menos un punto) y extraer todas las que aparecen en un texto.

In [8]:
EMAIL = r"[\w.+-]+@[A-Za-z\d-]+(?:\.[A-Za-z\d-]+)*\.[A-Za-z]{2,}"

def es_email(texto):
    return re.fullmatch(EMAIL, texto) is not None

casos = {"ana@example.com": True, "luis.perez+python@correo.ull.es": True, "sin-arroba.com": False,
         "dos@@example.com": False, "falta@dominio": False, "a@b.c": False}
for email, esperado in casos.items():
    print(f"{email:>31}: {es_email(email)}")
    assert es_email(email) == esperado

encontrados = re.findall(EMAIL, "Escriban a ana@example.com o a soporte@empresa.es (no a info@local).")
print(encontrados)
assert encontrados == ["ana@example.com", "soporte@empresa.es"]

                ana@example.com: True
luis.perez+python@correo.ull.es: True
                 sin-arroba.com: False
               dos@@example.com: False
                  falta@dominio: False
                          a@b.c: False
['ana@example.com', 'soporte@empresa.es']


## 9. Matrículas con `re.VERBOSE`

Reconocer matrículas españolas del sistema actual (4 dígitos y 3 consonantes, sin `Ñ` ni `Q`) y del provincial antiguo (`GC-1234-AB`), escribiendo el patrón comentado con `re.VERBOSE` y diciendo de qué tipo es cada una con `lastgroup`.

In [9]:
MATRICULA = re.compile(r"""
    (?P<actual>
        \d{4} [\s-]?
        [BCDFGHJKLMNPRSTVWXYZ]{3}    # sin vocales, Ñ ni Q
    )
  | (?P<provincial>
        [A-Z]{1,2} [\s-]?            # código de provincia
        \d{4} [\s-]?
        [A-Z]{1,2}
    )
""", re.VERBOSE)

def tipo_matricula(texto):
    m = MATRICULA.fullmatch(texto.upper())
    return m.lastgroup if m else None

casos = {"1234 BCD": "actual", "0000-zzz": "actual", "1234AEI": None, "GC-1234-AB": "provincial",
         "M 5678 Z": "provincial", "1234 BC": None}
for matricula, esperado in casos.items():
    print(f"{matricula:>10}: {tipo_matricula(matricula)}")
    assert tipo_matricula(matricula) == esperado

  1234 BCD: actual
  0000-zzz: actual
   1234AEI: None
GC-1234-AB: provincial
  M 5678 Z: provincial
   1234 BC: None


## 10. Extraer fechas de un texto

Extraer las fechas escritas como `dd/mm/aaaa`, `aaaa-mm-dd` o `5 de marzo de 2026`, convertirlas en `date`, ordenarlas por su posición y descartar las que no existen.

In [10]:
from datetime import date

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio",
         "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
PATRONES = [
    (r"\b(\d{1,2})/(\d{1,2})/(\d{4})\b", lambda d, m, a: (a, m, d)),
    (r"\b(\d{4})-(\d{2})-(\d{2})\b", lambda a, m, d: (a, m, d)),
    (r"\b(\d{1,2}) de (" + "|".join(MESES) + r") de (\d{4})\b", lambda d, m, a: (a, MESES.index(m.lower()) + 1, d)),
]

def extraer_fechas(texto):
    fechas = []
    for patron, ordenar in PATRONES:
        for m in re.finditer(patron, texto, re.IGNORECASE):
            a, mes, d = map(int, ordenar(*m.groups()))
            try:
                fechas.append((m.start(), date(a, mes, d)))
            except ValueError:
                print("Descartada:", m.group())
    return [f for _, f in sorted(fechas)]

texto = ("La matrícula abre el 15/09/2026 y cierra el 2026-10-01. El examen es el 3 de Diciembre de 2026, "
         "no el 31/02/2026.")
fechas = extraer_fechas(texto)
print(fechas)
assert fechas == [date(2026, 9, 15), date(2026, 10, 1), date(2026, 12, 3)]

Descartada: 31/02/2026
[datetime.date(2026, 9, 15), datetime.date(2026, 10, 1), datetime.date(2026, 12, 3)]


## 11. Contraseñas con *lookahead*

Validar con un único patrón con *lookaheads* que una contraseña tenga al menos 8 caracteres sin espacios, una minúscula, una mayúscula, un dígito y un símbolo, y compararlo con una comprobación sin `re` sobre 20 000 cadenas aleatorias.

In [11]:
import random
import string

SIMBOLOS = "!@#$%&*?-."
SEGURA = re.compile(r"(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[^\w\s])\S{8,}")

def es_segura(clave):
    return SEGURA.fullmatch(clave) is not None

def es_segura_sin_re(clave):
    return (len(clave) >= 8 and " " not in clave
            and any(c in string.ascii_lowercase for c in clave)
            and any(c in string.ascii_uppercase for c in clave)
            and any(c in string.digits for c in clave)
            and any(c in SIMBOLOS for c in clave))

random.seed(11)
alfabeto = string.ascii_letters + string.digits + SIMBOLOS + " "
for _ in range(20_000):
    clave = "".join(random.choices(alfabeto, k=random.randint(5, 12)))
    assert es_segura(clave) == es_segura_sin_re(clave), clave

for clave in ("Python2026!", "python2026!", "Corta1!", "SinSimbolo1A", "Con espacio 1A!"):
    print(f"{clave!r:>18}: {es_segura(clave)}")

     'Python2026!': True
     'python2026!': False
         'Corta1!': False
    'SinSimbolo1A': False
 'Con espacio 1A!': False


## 12. Tokenizar una expresión aritmética

Escribir `tokenizar(expresion)`, que divida una expresión en números, operadores y paréntesis con un único patrón de grupos con nombre y `lastgroup`, ignorando los espacios y avisando de los caracteres no válidos.

In [12]:
TOKEN = re.compile(r"""
    (?P<NUMERO>\d+(?:\.\d+)?)
  | (?P<OPERADOR>\*\*|[-+*/%])
  | (?P<PARENTESIS>[()])
  | (?P<ESPACIO>\s+)
  | (?P<ERROR>.)
""", re.VERBOSE)

def tokenizar(expresion):
    tokens = []
    for m in TOKEN.finditer(expresion):
        if m.lastgroup == "ERROR":
            raise ValueError(f"Carácter no válido {m.group()!r} en la posición {m.start()}")
        if m.lastgroup != "ESPACIO":
            tokens.append((m.lastgroup, m.group()))
    return tokens

expresion = "3.5 * (2 + 10) ** 2 - 7 % 4"
tokens = tokenizar(expresion)
print(tokens)
assert "".join(valor for _, valor in tokens) == expresion.replace(" ", "")
assert eval(" ".join(valor for _, valor in tokens)) == eval(expresion)

try:
    tokenizar("2 + x")
except ValueError as e:
    print(e)

[('NUMERO', '3.5'), ('OPERADOR', '*'), ('PARENTESIS', '('), ('NUMERO', '2'), ('OPERADOR', '+'), ('NUMERO', '10'), ('PARENTESIS', ')'), ('OPERADOR', '**'), ('NUMERO', '2'), ('OPERADOR', '-'), ('NUMERO', '7'), ('OPERADOR', '%'), ('NUMERO', '4')]
Carácter no válido 'x' en la posición 4
